<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ami-null/python-course/blob/main/09_SUPPLEMENTARY_oop.ipynb)

# OOP in Python: Supplementary Topics

## Learning Objectives

1. Understand the difference between class attributes and instance attributes
2. Know when to use a class attribute over an instance attribute
3. Distinguish between instance methods, class methods, and static methods
4. Use `@classmethod` to create alternative constructors
5. Use `@staticmethod` for utility functions scoped to a class
6. Understand name mangling with `__attribute` and when to use it
7. Use `@property` to expose controlled read access to attributes
8. Use `@setter` to add validation on attribute assignment
9. Define abstract base classes using the `abc` module
10. Enforce method implementation contracts across subclasses

## Introduction

The main lecture covered the core structure of OOP: classes, objects, inheritance, and polymorphism. This supplementary lecture goes one level deeper.

The topics here are not advanced for their own sake. Each one solves a specific practical problem:

| Topic | Problem it solves |
|---|---|
| Class attributes | Data that belongs to the class, not to any one instance |
| Class methods | Alternative ways to construct objects; operations on class-level data |
| Static methods | Utility functions that belong in a class but do not need instance or class data |
| Protected attributes | Communicating that an attribute is internal and should not be touched directly |
| `@property` | Controlled read access to attributes, with the clean syntax of attribute access |
| `@setter` | Validation logic on attribute assignment, without exposing raw attribute writes |
| Abstract Base Classes | Enforcing that every subclass implements a required set of methods |

---

## Class Attributes vs Instance Attributes

In the main lecture, every attribute was set inside `__init__` using `self.attribute = value`. These are **instance attributes**: each object gets its own copy.

A **class attribute** is defined directly in the class body, outside any method. It belongs to the class itself and is shared across all instances.

**When to prefer a class attribute over an instance attribute:**
- The value is the same for every instance of the class (e.g. number of wheels on a car)
- You want to track something across all instances (e.g. how many accounts have been created)
- The value is a constant or configuration that defines the class itself

**When to use an instance attribute:**
- The value is specific to each object (e.g. a car's color, an account's balance)
- The value changes over the lifetime of individual objects

In [ ]:
class Car:
    wheels = 4           # class attribute: every Car has 4 wheels
    total_cars = 0       # class attribute: tracks how many Car objects have been created

    def __init__(self, brand, model, year, color):
        self.brand = brand       # instance attribute: specific to this car
        self.model = model
        self.year = year
        self.color = color
        self.fuel_level = 100
        self.is_running = False
        Car.total_cars += 1      # update the class attribute whenever a new car is created

    def show_info(self):
        status = "running" if self.is_running else "off"
        return f"{self.year} {self.color} {self.brand} {self.model} | Fuel: {self.fuel_level} | Engine: {status}"

In [ ]:
# before any objects are created
print(Car.total_cars)

0


In [ ]:
car1 = Car("Toyota", "Corolla", 2020, "red")
car2 = Car("BMW", "M3", 2022, "blue")

print(Car.total_cars)     # accessed on the class
print(car1.total_cars)    # also accessible on an instance, but it is still the class's value

2
2


In [ ]:
# class attributes are accessible on all instances
print(car1.wheels)
print(car2.wheels)

4
4


### What Happens When You Assign to a Class Attribute on an Instance

This is a common source of confusion. If you write `car1.wheels = 3`, Python does **not** modify the class attribute. Instead, it creates a new instance attribute on `car1` that shadows the class attribute. The class attribute itself is untouched.

In [ ]:
car1.wheels = 3    # creates a new instance attribute on car1, does not change Car.wheels

print(car1.wheels)    # 3: reads from the instance attribute (shadows the class attribute)
print(car2.wheels)    # 4: unaffected, still reads from the class attribute
print(Car.wheels)     # 4: the class attribute is unchanged

3
4
4


In [ ]:
# to modify the class attribute for all instances, assign on the class itself
Car.wheels = 6

print(car1.wheels)    # still 3: car1's instance attribute takes priority
print(car2.wheels)    # 6: car2 has no instance attribute, so reads from Car
print(Car.wheels)     # 6

3
6
6


The rule Python uses when looking up an attribute on an instance is: check the instance first, then the class. This is why the instance attribute "shadows" the class attribute once it exists.

---

## Instance Methods, Class Methods, and Static Methods

Python has three kinds of methods you can define inside a class:

| Method type | First parameter | Decorator | Operates on |
|---|---|---|---|
| Instance method | `self` | none | The specific instance |
| Class method | `cls` | `@classmethod` | The class itself |
| Static method | neither `self` nor `cls` | `@staticmethod` | Neither; just a scoped function |

### Instance Methods

These are the methods you already know. They receive `self` as the first argument and can read or modify the instance's attributes.

In [ ]:
class BankAccount:
    total_accounts = 0    # class attribute: shared across all instances

    def __init__(self, account_holder, balance=0):
        self.account_holder = account_holder
        self.balance = balance
        BankAccount.total_accounts += 1

    # instance method: operates on self
    def deposit(self, amount):
        if amount <= 0:
            return "Deposit amount must be positive."
        self.balance += amount
        return f"Deposited {amount}. New balance: {self.balance}"

    def withdraw(self, amount):
        if amount <= 0:
            return "Withdrawal amount must be positive."
        if amount > self.balance:
            return "Insufficient funds."
        self.balance -= amount
        return f"Withdrew {amount}. New balance: {self.balance}"

    def show_balance(self):
        return f"[{self.account_holder}] Balance: {self.balance}"

### Class Methods

A class method receives the class as its first argument (`cls`) instead of the instance. It is defined with the `@classmethod` decorator.

The most common use case is as an **alternative constructor**: a factory method that creates instances in a way that `__init__` alone cannot express cleanly.

In [ ]:
class BankAccount:
    total_accounts = 0

    def __init__(self, account_holder, balance=0):
        self.account_holder = account_holder
        self.balance = balance
        BankAccount.total_accounts += 1

    @classmethod
    def get_total_accounts(cls):
        # operates on class-level data, not on any particular instance
        return f"Total accounts opened: {cls.total_accounts}"

    @classmethod
    def from_dict(cls, data):
        # alternative constructor: creates an account from a dictionary
        # useful when data arrives from an API or a database as a dict
        return cls(data["holder"], data["balance"])

    def deposit(self, amount):
        if amount <= 0:
            return "Deposit amount must be positive."
        self.balance += amount
        return f"Deposited {amount}. New balance: {self.balance}"

    def withdraw(self, amount):
        if amount <= 0:
            return "Withdrawal amount must be positive."
        if amount > self.balance:
            return "Insufficient funds."
        self.balance -= amount
        return f"Withdrew {amount}. New balance: {self.balance}"

    def show_balance(self):
        return f"[{self.account_holder}] Balance: {self.balance}"

In [ ]:
# calling a class method on the class itself
print(BankAccount.get_total_accounts())

Total accounts opened: 0


In [ ]:
acc1 = BankAccount("Alice", 5000)
acc2 = BankAccount("Bob", 2000)

print(BankAccount.get_total_accounts())

Total accounts opened: 2


In [ ]:
# using the alternative constructor
data = {"holder": "Charlie", "balance": 1500}
acc3 = BankAccount.from_dict(data)

acc3.show_balance()

'[Charlie] Balance: 1500'

In [ ]:
# class methods can also be called on an instance, though calling on the class is more conventional
print(acc1.get_total_accounts())

Total accounts opened: 3


Python's built-in `int` uses this pattern: `int.from_bytes(b'\x00\x10', 'big')` is a class method that constructs an integer from bytes rather than from a plain number.

### Static Methods

A static method receives neither `self` nor `cls`. It is a plain function that lives inside the class namespace because it is logically related to the class, but it does not need access to any instance or class data.

Use a static method when the function only uses the arguments passed to it and has no reason to touch `self` or `cls`.

In [ ]:
class BankAccount:
    total_accounts = 0

    def __init__(self, account_holder, balance=0):
        self.account_holder = account_holder
        self.balance = balance
        BankAccount.total_accounts += 1

    @classmethod
    def get_total_accounts(cls):
        return f"Total accounts opened: {cls.total_accounts}"

    @classmethod
    def from_dict(cls, data):
        return cls(data["holder"], data["balance"])

    @staticmethod
    def is_valid_amount(amount):
        # purely a validation helper: only uses its argument, needs no instance or class
        return isinstance(amount, (int, float)) and amount > 0

    @staticmethod
    def currency_symbol(currency_code):
        # a lookup utility logically related to banking, but not tied to any account
        symbols = {"USD": "$", "EUR": "€", "GBP": "£", "BDT": "৳"}
        return symbols.get(currency_code, "?")

    def deposit(self, amount):
        if not BankAccount.is_valid_amount(amount):    # reusing the static method inside the class
            return "Deposit amount must be a positive number."
        self.balance += amount
        return f"Deposited {amount}. New balance: {self.balance}"

    def withdraw(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Withdrawal amount must be a positive number."
        if amount > self.balance:
            return "Insufficient funds."
        self.balance -= amount
        return f"Withdrew {amount}. New balance: {self.balance}"

    def show_balance(self):
        return f"[{self.account_holder}] Balance: {self.balance}"

In [ ]:
# static methods are called on the class
print(BankAccount.is_valid_amount(500))
print(BankAccount.is_valid_amount(-10))
print(BankAccount.is_valid_amount("hello"))

True
False
False


In [ ]:
print(BankAccount.currency_symbol("BDT"))
print(BankAccount.currency_symbol("EUR"))

৳
€


In [ ]:
# the validation now runs inside deposit and withdraw
acc = BankAccount("Alice", 1000)
print(acc.deposit("five hundred"))
print(acc.deposit(500))

Deposit amount must be a positive number.
Deposited 500. New balance: 1500


**When to prefer each method type:**

| You need access to... | Use |
|---|---|
| Instance data (`self.balance`, `self.name`) | Instance method |
| Class data (`cls.total_accounts`) or alternate construction | Class method |
| Neither; purely the arguments passed in | Static method |

If you find yourself writing a static method that has no logical connection to the class, it is probably better off as a standalone module-level function.

---

## Protected Attributes

Python does not have true access control. There is no keyword that prevents code outside the class from reading or writing an attribute. However, Python has a convention and a mechanism that signal "do not touch this directly".

### Name Mangling with `__attribute`

When you prefix an attribute name with two underscores (e.g. `__balance`), Python applies **name mangling**: it renames the attribute internally to `_ClassName__attribute`. This is not a lock, but it does make accidental access from outside significantly less likely.

The intent is: this attribute is part of the internal implementation of this class. Subclasses and outside code should not rely on it directly.

In [ ]:
class BankAccount:
    total_accounts = 0

    def __init__(self, account_holder, balance=0):
        self.account_holder = account_holder
        self.__balance = balance          # name-mangled: stored as _BankAccount__balance
        BankAccount.total_accounts += 1

    def deposit(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Deposit amount must be a positive number."
        self.__balance += amount
        return f"Deposited {amount}. New balance: {self.__balance}"

    def withdraw(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Withdrawal amount must be a positive number."
        if amount > self.__balance:
            return "Insufficient funds."
        self.__balance -= amount
        return f"Withdrew {amount}. New balance: {self.__balance}"

    def show_balance(self):
        return f"[{self.account_holder}] Balance: {self.__balance}"

    @classmethod
    def get_total_accounts(cls):
        return f"Total accounts opened: {cls.total_accounts}"

    @classmethod
    def from_dict(cls, data):
        return cls(data["holder"], data["balance"])

    @staticmethod
    def is_valid_amount(amount):
        return isinstance(amount, (int, float)) and amount > 0

In [ ]:
acc = BankAccount("Alice", 1000)
acc.show_balance()

'[Alice] Balance: 1000'

In [ ]:
# UNCOMMENT TO SEE THE ERROR: __balance is not accessible by its original name
# print(acc.__balance)

In [ ]:
# name mangling stores it as _BankAccount__balance
# this is accessible, but the mangled name signals: you are not supposed to be here
print(acc._BankAccount__balance)

1000


The mangled name `_BankAccount__balance` is accessible if you really want it. Python does not enforce privacy. The mechanism exists to prevent accidental name collisions in inheritance hierarchies and to communicate intent: this is internal.

**The convention in Python:**

| Naming | Convention | Meaning |
|---|---|---|
| `attribute` | Public | Intended for general use |
| `__attribute` | Protected (name-mangled) | Internal; do not access from outside |

---
## Properties: `@property` and `@setter`

Once `__balance` is protected, outside code can no longer read it with `acc.balance`. If you want to allow reading it but not raw writing, you need a **property**.

A `@property` lets you expose a method as if it were an attribute. The caller writes `acc.balance`, not `acc.balance()`, but under the hood Python calls the method.

A `@setter` lets you intercept assignment (`acc.balance = 500`) and run validation before the value is stored.

**When to use `@property`:**
- You want read access to a protected attribute without exposing it as a raw attribute
- The "attribute" is actually computed from other data
- You want to add validation logic on assignment
- You want a clean public interface that does not expose implementation details

In [ ]:
class BankAccount:
    total_accounts = 0

    def __init__(self, account_holder, balance=0):
        self.account_holder = account_holder
        self.__balance = balance
        BankAccount.total_accounts += 1

    @property
    def balance(self):
        # this method is called when someone reads acc.balance
        return self.__balance

    @balance.setter
    def balance(self, value):
        # this method is called when someone writes acc.balance = value
        # we validate before allowing the change
        if not isinstance(value, (int, float)):
            raise TypeError("Balance must be a number.")
        if value < 0:
            raise ValueError("Balance cannot be negative.")
        self.__balance = value

    def deposit(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Deposit amount must be a positive number."
        self.__balance += amount
        return f"Deposited {amount}. New balance: {self.__balance}"

    def withdraw(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Withdrawal amount must be a positive number."
        if amount > self.__balance:
            return "Insufficient funds."
        self.__balance -= amount
        return f"Withdrew {amount}. New balance: {self.__balance}"

    def show_balance(self):
        return f"[{self.account_holder}] Balance: {self.__balance}"

    @classmethod
    def get_total_accounts(cls):
        return f"Total accounts opened: {cls.total_accounts}"

    @classmethod
    def from_dict(cls, data):
        return cls(data["holder"], data["balance"])

    @staticmethod
    def is_valid_amount(amount):
        return isinstance(amount, (int, float)) and amount > 0

In [ ]:
acc = BankAccount("Alice", 1000)

# reading balance: calls the @property getter, looks like plain attribute access
print(acc.balance)

1000


In [ ]:
# writing balance: calls the @setter, which validates before storing
acc.balance = 2000
print(acc.balance)

2000


In [ ]:
# UNCOMMENT TO SEE THE ERROR: setter rejects negative values
# acc.balance = -500

In [ ]:
# UNCOMMENT TO SEE THE ERROR: setter rejects non-numeric values
# acc.balance = "a lot"

### A Read-Only Property

If you define `@property` without a corresponding `@setter`, the attribute becomes read-only. Attempting to assign to it raises an `AttributeError`.

This is useful for computed values that should never be set directly.

In [ ]:
class Car:
    wheels = 4
    total_cars = 0

    def __init__(self, brand, model, year, color):
        self.brand = brand
        self.model = model
        self.year = year
        self.color = color
        self.__fuel_level = 100
        self.is_running = False
        Car.total_cars += 1

    @property
    def fuel_level(self):
        return self.__fuel_level

    # no @fuel_level.setter defined: fuel_level is read-only from outside
    # the only way to change fuel is through refuel() or drive()

    def refuel(self, amount):
        self.__fuel_level = min(100, self.__fuel_level + amount)
        return f"Refueled. Fuel level: {self.__fuel_level}"

    def start_engine(self):
        if self.is_running:
            return f"{self.brand} {self.model} is already running."
        self.is_running = True
        return f"{self.brand} {self.model} engine started."

    def drive(self, distance):
        if not self.is_running:
            return "Start the engine first."
        if distance > self.__fuel_level:
            return "Not enough fuel for that distance."
        self.__fuel_level -= distance
        return f"Drove {distance} km. Fuel remaining: {self.__fuel_level}"

    def show_info(self):
        status = "running" if self.is_running else "off"
        return f"{self.year} {self.color} {self.brand} {self.model} | Fuel: {self.__fuel_level} | Engine: {status}"

In [ ]:
car = Car("Toyota", "Corolla", 2020, "red")

# reading fuel_level is allowed
print(car.fuel_level)

100


In [ ]:
# UNCOMMENT TO SEE THE ERROR: no setter defined, so direct assignment is blocked
# car.fuel_level = 999

In [ ]:
# the only way to change fuel is through the intended interface
car.start_engine()
car.drive(40)
print(car.fuel_level)

60


### `@property` and Computed Attributes

A property does not have to return a stored value. It can compute a value on the fly from other attributes. To the caller, it still looks like a plain attribute access.

In [ ]:
class BankAccount:
    total_accounts = 0

    def __init__(self, account_holder, balance=0, overdraft_limit=0):
        self.account_holder = account_holder
        self.__balance = balance
        self.overdraft_limit = overdraft_limit
        BankAccount.total_accounts += 1

    @property
    def balance(self):
        return self.__balance

    @balance.setter
    def balance(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("Balance must be a number.")
        if value < 0:
            raise ValueError("Balance cannot be negative.")
        self.__balance = value

    @property
    def available_funds(self):
        # computed from two other attributes: no separate stored value needed
        return self.__balance + self.overdraft_limit

    def deposit(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Deposit amount must be a positive number."
        self.__balance += amount
        return f"Deposited {amount}. New balance: {self.__balance}"

    def withdraw(self, amount):
        if not BankAccount.is_valid_amount(amount):
            return "Withdrawal amount must be a positive number."
        if amount > self.available_funds:    # uses the computed property
            return "Insufficient funds."
        self.__balance -= amount
        return f"Withdrew {amount}. New balance: {self.__balance}"

    def show_balance(self):
        return f"[{self.account_holder}] Balance: {self.__balance} | Available: {self.available_funds}"

    @classmethod
    def get_total_accounts(cls):
        return f"Total accounts opened: {cls.total_accounts}"

    @classmethod
    def from_dict(cls, data):
        return cls(data["holder"], data["balance"])

    @staticmethod
    def is_valid_amount(amount):
        return isinstance(amount, (int, float)) and amount > 0

In [ ]:
acc = BankAccount("Alice", balance=1000, overdraft_limit=500)
print(acc.show_balance())

[Alice] Balance: 1000 | Available: 1500


In [ ]:
# available_funds is computed, not stored
print(acc.available_funds)

1500


In [ ]:
# withdrawal up to available_funds (balance + overdraft) is allowed
acc.withdraw(1400)

'Withdrew 1400. New balance: -400'

In [ ]:
acc.show_balance()

'[Alice] Balance: -400 | Available: 100'

---

## Abstract Base Classes

In the main lecture, inheritance let subclasses override methods. But nothing stopped a subclass from simply not implementing an expected method. The code would only break at runtime when the method was called.

An **Abstract Base Class (ABC)** solves this by making it a contract: if a subclass does not implement a required method, Python will refuse to instantiate it at all. The error is caught earlier, at instantiation time rather than at call time.

**When to use an ABC:**
- You are designing a class hierarchy where every subclass must implement a specific set of methods
- You want to define an interface without providing a default implementation
- You want Python to enforce that contract rather than relying on documentation or convention

**Syntax:**
- Import `ABC` and `abstractmethod` from the `abc` module
- Inherit from `ABC`
- Mark required methods with `@abstractmethod`

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):    # inheriting from ABC marks this as an abstract class

    @abstractmethod
    def area(self):
        # no implementation here: subclasses must provide their own
        pass

    @abstractmethod
    def perimeter(self):
        pass

    def describe(self):    # non-abstract: subclasses inherit this as-is
        return f"{type(self).__name__} | Area: {self.area():.2f} | Perimeter: {self.perimeter():.2f}"

In [ ]:
# UNCOMMENT TO SEE THE ERROR: you cannot instantiate an abstract class directly
# s = Shape()

In [ ]:
class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2

    def perimeter(self):
        return 2 * 3.14159 * self.radius


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

In [ ]:
c = Circle(5)
r = Rectangle(4, 6)

print(c.describe())
print(r.describe())

Circle | Area: 78.54 | Perimeter: 31.42
Rectangle | Area: 24.00 | Perimeter: 20.00


In [ ]:
# polymorphism: same interface, different behavior
shapes = [Circle(3), Rectangle(2, 5), Circle(7)]

for shape in shapes:
    print(shape.describe())

Circle | Area: 28.27 | Perimeter: 18.85
Rectangle | Area: 10.00 | Perimeter: 14.00
Circle | Area: 153.94 | Perimeter: 43.98


### What Happens If a Subclass Does Not Implement All Abstract Methods

In [ ]:
class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return 0.5 * self.base * self.height

    # perimeter() is not implemented

# UNCOMMENT TO SEE THE ERROR: Python refuses to instantiate because perimeter() is not implemented
# t = Triangle(3, 4)

### ABCs with `BankAccount`

We can apply the same pattern to the bank account hierarchy. An abstract `BaseAccount` defines what every account type must be able to do, without dictating how.

In [ ]:
from abc import ABC, abstractmethod

class BaseAccount(ABC):

    @abstractmethod
    def deposit(self, amount):
        pass

    @abstractmethod
    def withdraw(self, amount):
        pass

    @abstractmethod
    def show_balance(self):
        pass


class BankAccount(BaseAccount):    # now inherits from BaseAccount instead of object
    total_accounts = 0

    def __init__(self, account_holder, balance=0, overdraft_limit=0):
        self.account_holder = account_holder
        self.__balance = balance
        self.overdraft_limit = overdraft_limit
        BankAccount.total_accounts += 1

    @property
    def balance(self):
        return self.__balance

    @balance.setter
    def balance(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("Balance must be a number.")
        if value < 0:
            raise ValueError("Balance cannot be negative.")
        self.__balance = value

    @property
    def available_funds(self):
        return self.__balance + self.overdraft_limit

    def deposit(self, amount):    # satisfies the abstract method requirement
        if not BankAccount.is_valid_amount(amount):
            return "Deposit amount must be a positive number."
        self.__balance += amount
        return f"Deposited {amount}. New balance: {self.__balance}"

    def withdraw(self, amount):    # satisfies the abstract method requirement
        if not BankAccount.is_valid_amount(amount):
            return "Withdrawal amount must be a positive number."
        if amount > self.available_funds:
            return "Insufficient funds."
        self.__balance -= amount
        return f"Withdrew {amount}. New balance: {self.__balance}"

    def show_balance(self):    # satisfies the abstract method requirement
        return f"[{self.account_holder}] Balance: {self.__balance} | Available: {self.available_funds}"

    @classmethod
    def get_total_accounts(cls):
        return f"Total accounts opened: {cls.total_accounts}"

    @classmethod
    def from_dict(cls, data):
        return cls(data["holder"], data["balance"])

    @staticmethod
    def is_valid_amount(amount):
        return isinstance(amount, (int, float)) and amount > 0

In [ ]:
acc = BankAccount("Alice", 1000)
print(acc.show_balance())
print(acc.deposit(500))
print(acc.withdraw(200))

[Alice] Balance: 1000 | Available: 1000
Deposited 500. New balance: 1500
Withdrew 200. New balance: 1300


In [ ]:
# isinstance still works: BankAccount is a BaseAccount
print(isinstance(acc, BaseAccount))
print(isinstance(acc, BankAccount))

True
True


## Exercises

**Exercise 1**

Add a class attribute `total_cars` to the `Car` class that tracks how many `Car` objects have been instantiated. Add a class method `get_fleet_size()` that returns the current count. Create three `Car` objects and confirm the count is correct.

**Exercise 2**

Add a static method `is_valid_year(year)` to the `Car` class that returns `True` if the year is between 1886 (the year the first car was invented) and the current year, and `False` otherwise. Use this inside `__init__` to raise a `ValueError` if an invalid year is passed.

**Exercise 3**

Add a class method `from_string(cls, car_string)` to `Car` that acts as an alternative constructor. It should accept a string in the format `"Toyota,Corolla,2020,red"` and return a `Car` object. Test it by creating a car from a string.

**Exercise 4**

Rewrite the `Car` class so that `fuel_level` is a protected attribute (`__fuel_level`). Add a `@property` to expose it as read-only. Add a `@property` for `color` with a `@setter` that rejects any value that is not a string.

**Exercise 5**

Define an abstract base class called `Vehicle` with three abstract methods: `start()`, `stop()`, and `fuel_status()`. Create two concrete subclasses: `Car` and `ElectricScooter`. `Car` should use fuel level as its fuel status. `ElectricScooter` should use battery percentage. Both must implement all three methods.

**Exercise 6**

Define an abstract base class `BaseAccount` with abstract methods `deposit()`, `withdraw()`, and `show_balance()`. Create two concrete subclasses: `SavingsAccount` and `FixedDepositAccount`.

`SavingsAccount` should:
- Use a protected `__balance` with a `@property` and `@setter`
- Have an `interest_rate` class attribute (e.g. `0.04`)
- Add an `apply_interest()` method

`FixedDepositAccount` should:
- Accept a `term_months` argument on creation
- Override `withdraw()` to reject all withdrawals with a message saying the deposit is locked until the term ends

Use `isinstance` and `issubclass` to confirm both are instances of `BaseAccount`.